# Biometric Authentication — Lab 1
## Setup and Biometric Performance Metrics

| | |
|---|---|
| **Student name** | *(write your name here)* |
| **Student ID** | *(write your ID here)* |
| **Date** | *(write the date here)* |
| **Duration** | about 2 hours |
| **Hardware needed** | none — all data is generated by code |

**How to use this notebook:** read each text cell, then run the code cell below it (**Shift + Enter**). Run the cells **in order from top to bottom**. If something breaks, use **Runtime → Restart and run all**.

Cells marked **✏️ YOUR TURN** need you to write something. Double-click the cell to edit it.

## 1. Objectives

By the end of this lab you will be able to:

- Explain genuine and impostor comparisons, similarity scores, and decision thresholds
- Compute **FAR, FRR, GAR and EER** from score data
- Plot and interpret score distributions, FAR/FRR curves, ROC curves and DET curves
- Choose a decision threshold for a given security requirement and justify the trade-off
- Submit your work through a GitHub repository

## 2. Background: Key Terms

Every biometric system compares a new sample with a stored template and produces a **similarity score**. If the score is **at or above a threshold**, the person is accepted; otherwise rejected.

| Term | Meaning | How we compute it |
|---|---|---|
| **Genuine attempt** | A real user compared with their own template | scores in `genuine_scores.txt` |
| **Impostor attempt** | A person compared with someone else's template | scores in `impostor_scores.txt` |
| **Threshold (t)** | Accept if score ≥ t | chosen by the designer |
| **FAR** (False Accept Rate) | Fraction of impostors wrongly accepted | count(impostor ≥ t) / total impostor |
| **FRR** (False Reject Rate) | Fraction of genuine users wrongly rejected | count(genuine < t) / total genuine |
| **GAR** (Genuine Accept Rate) | Fraction of genuine users correctly accepted | 1 − FRR |
| **EER** (Equal Error Rate) | Error rate where FAR = FRR (lower is better) | find t where \|FAR − FRR\| is smallest |

> **Key idea:** raising the threshold makes the system stricter — FAR goes **down**, FRR goes **up**. Lowering it does the opposite. When the score distributions overlap, no threshold gives zero errors.

## Part A. Check Your Environment

**Step 1.** Run the cell below. You should see three version numbers and no error. Google Colab already has all three libraries installed.

*(Using your own laptop? Install with `pip install numpy matplotlib scikit-learn jupyter`.)*

In [ ]:
import numpy as np
import matplotlib
import sklearn

print("NumPy       :", np.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## Part B. Create Your GitHub Repository

**Step 2.** Do this in your browser (not in this notebook):

1. Go to **github.com** and sign in (create a free account if needed).
2. Click **+** (top right) → **New repository**.
3. Repository name: `biometric-auth-lab1-YOURID` (replace `YOURID` with your student ID).
4. Set visibility to **Public** and tick **Add a README file**.
5. Click **Create repository**.

Keep the tab open — you will upload your work in **Part J**.

## Part C. Generate the Score Data

Real systems produce scores by comparing fingerprints, faces, etc. To focus on the *metrics*, we generate realistic scores with code. Genuine scores are higher on average than impostor scores, but the two **overlap**.

**Step 3.** Run the cell below.

In [ ]:
rng = np.random.default_rng(42)   # fixed seed: everyone gets the same data

# Genuine scores: 1000 comparisons, centred around 0.70
genuine  = np.clip(rng.normal(loc=0.70, scale=0.10, size=1000), 0, 1)

# Impostor scores: 5000 comparisons, centred around 0.40
impostor = np.clip(rng.normal(loc=0.40, scale=0.12, size=5000), 0, 1)

np.savetxt("genuine_scores.txt", genuine)
np.savetxt("impostor_scores.txt", impostor)
print("Files saved.")

**Expected output:** `Files saved.`

**Step 4.** Confirm the files exist. In Colab, click the 📁 folder icon on the left — you should see `genuine_scores.txt` and `impostor_scores.txt`.

## Part D. Load and Explore the Scores

**Step 5.** Load the two files back into memory and check sizes and averages.

In [ ]:
genuine  = np.loadtxt("genuine_scores.txt")
impostor = np.loadtxt("impostor_scores.txt")

print("Genuine :", genuine.shape,  "mean =", round(genuine.mean(), 3))
print("Impostor:", impostor.shape, "mean =", round(impostor.mean(), 3))

**Expected output:**
```
Genuine : (1000,) mean = 0.697
Impostor: (5000,) mean = 0.4
```

**Step 6.** Plot both score distributions on one graph.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(impostor, bins=40, alpha=0.6, label="Impostor", density=True)
plt.hist(genuine,  bins=40, alpha=0.6, label="Genuine",  density=True)
plt.xlabel("Similarity score")
plt.ylabel("Density")
plt.title("Genuine vs Impostor score distributions")
plt.legend()
plt.savefig("fig1_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

The figure is saved as `fig1_distributions.png`. Notice that the two humps **overlap** in the middle — that overlap is exactly why biometric errors exist.

### ✏️ YOUR TURN — Observation 1
*In one or two sentences, describe where the two distributions overlap and what that means for a system that must decide accept/reject.*

**Your answer:** *(double-click to write here)*

## Part E. Compute FAR and FRR at One Threshold

**Step 7.** Write a function that applies a threshold and counts the errors, then test it at `t = 0.55`.

- `impostor >= threshold` is `True` for every impostor who would be **accepted**. `np.mean` of True/False values gives the fraction of Trues → that is the **FAR**.
- FRR is the same idea for genuine scores **below** the threshold.

In [ ]:
def far_frr_at(threshold, genuine, impostor):
    far = np.mean(impostor >= threshold)   # impostors wrongly ACCEPTED
    frr = np.mean(genuine  <  threshold)   # genuine users wrongly REJECTED
    return far, frr

far, frr = far_frr_at(0.55, genuine, impostor)
print(f"Threshold 0.55 -> FAR = {far:.2%}, FRR = {frr:.2%}")

**Expected output:** `Threshold 0.55 -> FAR = 10.16%, FRR = 6.80%`

**Step 8.** Try four different thresholds.

In [ ]:
for t in [0.45, 0.55, 0.65, 0.75]:
    far, frr = far_frr_at(t, genuine, impostor)
    print(f"t = {t:.2f}  FAR = {far:.2%}  FRR = {frr:.2%}")

**Expected output:**
```
t = 0.45  FAR = 33.46%  FRR = 0.70%
t = 0.55  FAR = 10.16%  FRR = 6.80%
t = 0.65  FAR = 1.92%  FRR = 29.90%
t = 0.75  FAR = 0.20%  FRR = 72.50%
```

### ✏️ YOUR TURN — Observation 2
*What happens to FAR and FRR as the threshold increases? Use numbers from the output above.*

**Your answer:** *(double-click to write here)*

## Part F. Sweep All Thresholds and Find the EER

**Step 9.** Compute FAR and FRR for 1001 thresholds between 0 and 1, and locate the point where they are equal (the **EER**).

In [ ]:
thresholds = np.linspace(0, 1, 1001)      # 0.000, 0.001, ... 1.000

far_list, frr_list = [], []
for t in thresholds:
    far, frr = far_frr_at(t, genuine, impostor)
    far_list.append(far)
    frr_list.append(frr)

far_arr = np.array(far_list)
frr_arr = np.array(frr_list)

# EER: the threshold where FAR and FRR are closest to each other
idx = np.argmin(np.abs(far_arr - frr_arr))
eer = (far_arr[idx] + frr_arr[idx]) / 2
eer_threshold = thresholds[idx]
print(f"EER = {eer:.2%} at threshold {eer_threshold:.3f}")

**Expected output:** `EER = 8.81% at threshold 0.562`

**Step 10.** Plot FAR and FRR against the threshold.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(thresholds, far_arr, label="FAR")
plt.plot(thresholds, frr_arr, label="FRR")
plt.axvline(eer_threshold, linestyle="--", color="gray",
            label=f"EER = {eer:.2%}")
plt.xlabel("Threshold")
plt.ylabel("Error rate")
plt.title("FAR and FRR versus threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig2_far_frr.png", dpi=150, bbox_inches="tight")
plt.show()

The two curves cross at the EER point (dashed line). The figure is saved as `fig2_far_frr.png`.

## Part G. ROC and DET Curves

**Step 11.** Draw the ROC and DET curves and compute the **AUC** (area under the ROC curve).

In [ ]:
from sklearn.metrics import roc_curve, auc

# Build labels: 1 = genuine, 0 = impostor
labels = np.concatenate([np.ones(len(genuine)), np.zeros(len(impostor))])
scores = np.concatenate([genuine, impostor])

fpr, tpr, _ = roc_curve(labels, scores)      # fpr = FAR, tpr = GAR
print(f"AUC = {auc(fpr, tpr):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ROC curve: GAR (= 1 - FRR) versus FAR
axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)   # random-guess line
axes[0].set_xlabel("FAR")
axes[0].set_ylabel("GAR (1 - FRR)")
axes[0].set_title("ROC curve")

# DET curve: FRR versus FAR on log-log axes
fnr = 1 - tpr
axes[1].loglog(fpr + 1e-5, fnr + 1e-5)           # +1e-5 avoids log(0)
axes[1].set_xlabel("FAR")
axes[1].set_ylabel("FRR")
axes[1].set_title("DET curve (log-log)")
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("fig3_roc_det.png", dpi=150, bbox_inches="tight")
plt.show()

**Expected output:** `AUC = 0.9712`

- **ROC curve:** the closer it hugs the top-left corner, the better. The dashed diagonal is random guessing.
- **DET curve:** the same trade-off on log axes, which makes low error rates easier to read. Lower and further left is better.
- **AUC:** 1.0 = perfect separation; 0.5 = no better than a coin flip.

## Part H. Choose a Threshold for Real Requirements

Different applications need different trade-offs.

**Step 12.** Find the threshold for two scenarios:
- **High security:** FAR must be at most 0.1%
- **Convenience:** FRR must be at most 1%

In [ ]:
# Scenario 1: HIGH SECURITY - FAR must be at most 0.1%
ok = np.where(far_arr <= 0.001)[0]
t_sec = thresholds[ok[0]]          # lowest threshold that meets the target
print(f"High security : threshold = {t_sec:.3f}, "
      f"FAR = {far_arr[ok[0]]:.2%}, FRR = {frr_arr[ok[0]]:.2%}")

# Scenario 2: CONVENIENCE - FRR must be at most 1%
ok = np.where(frr_arr <= 0.01)[0]
t_conv = thresholds[ok[-1]]        # highest threshold that meets the target
print(f"Convenience   : threshold = {t_conv:.3f}, "
      f"FAR = {far_arr[ok[-1]]:.2%}, FRR = {frr_arr[ok[-1]]:.2%}")

**Expected output:**
```
High security : threshold = 0.771, FAR = 0.10%, FRR = 78.20%
Convenience   : threshold = 0.466, FAR = 28.90%, FRR = 1.00%
```

> **Think about it:** with this data, a very strict system rejects most genuine users, and a very convenient one accepts many impostors. In a real project, engineers would improve the matcher (better features, or fusion in Lab 7) rather than accept such a trade-off.

## Part I. Experiment: What If the Distributions Overlap More?

**Step 13.** Keep the genuine scores fixed and move the impostor average closer to the genuine average. Observe how the EER changes.

In [ ]:
def compute_eer(genuine, impostor):
    thresholds = np.linspace(0, 1, 1001)
    far = np.array([np.mean(impostor >= t) for t in thresholds])
    frr = np.array([np.mean(genuine  <  t) for t in thresholds])
    i = np.argmin(np.abs(far - frr))
    return (far[i] + frr[i]) / 2, thresholds[i]

rng = np.random.default_rng(7)
for mu in [0.30, 0.40, 0.50, 0.55, 0.60]:
    imp = np.clip(rng.normal(mu, 0.12, 5000), 0, 1)
    e, t = compute_eer(genuine, imp)
    print(f"Impostor mean = {mu:.2f} -> EER = {e:.2%} (threshold {t:.3f})")

**Expected output** (last decimal may differ slightly with library versions):
```
Impostor mean = 0.30 -> EER = 3.57% (threshold 0.517)
Impostor mean = 0.40 -> EER = 8.73% (threshold 0.560)
Impostor mean = 0.50 -> EER = 18.52% (threshold 0.607)
Impostor mean = 0.55 -> EER = 25.23% (threshold 0.631)
Impostor mean = 0.60 -> EER = 31.71% (threshold 0.655)
```

### ✏️ YOUR TURN — Observation 3
*Explain the trend in one or two sentences.*

**Your answer:** *(double-click to write here)*

## Part J. Summary of Your Results

**Step 14.** Run this cell to print a summary. Copy these numbers into the results table of your `README.md`.

In [ ]:
print("===== LAB 1 RESULTS SUMMARY =====")
print(f"EER                      : {eer:.2%} at threshold {eer_threshold:.3f}")
print(f"AUC                      : {auc(fpr, tpr):.4f}")
print(f"High-security threshold  : {t_sec:.3f}")
print(f"Convenience threshold    : {t_conv:.3f}")

## Part K. Submit Your Work on GitHub

**Step 15.** Save your files.
- In Colab: **File → Download → Download .ipynb** (make sure all cells have been run so outputs are visible).
- Open the 📁 folder panel on the left and download `fig1_distributions.png`, `fig2_far_frr.png`, `fig3_roc_det.png`, `genuine_scores.txt`, `impostor_scores.txt` (right-click → Download).

**Step 16.** Upload to your repository. Choose one method:

**Method A — Browser upload (easiest)**
1. Open your repository on github.com → **Add file → Upload files**.
2. Drag in the notebook, write commit message `Add Lab 1 notebook`, click **Commit changes**.
3. Repeat for the three figures (`Add figures`) and the two `.txt` files (`Add score files`).

**Method B — Git command line** (use a GitHub *Personal Access Token* when asked for a password)
```bash
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"

git clone https://github.com/YOUR-USERNAME/biometric-auth-lab1-YOUR-ID.git
cd biometric-auth-lab1-YOUR-ID
# copy your files into this folder, then:
git add Lab1_Biometric_Metrics.ipynb
git commit -m "Add Lab 1 notebook"
git add fig1_distributions.png fig2_far_frr.png fig3_roc_det.png
git commit -m "Add figures"
git add genuine_scores.txt impostor_scores.txt
git commit -m "Add score files"
git push origin main
```
*(Colab also offers **File → Save a copy in GitHub**.)*

> **Required:** at least **3 separate commits** with clear messages.

**Step 17.** Edit `README.md` on GitHub (pencil icon) and add: your name and ID, the results from Part J, and your answers to the analysis questions below.

**Step 18.** Submit your repository link through the course portal before the deadline. Open the link in a private browser window first to check it works.

## Analysis Questions

Answer each in **2–4 sentences, in your own words** (double-click the answer lines to write). Also copy them to your `README.md`.

**Q1.** What is the difference between FAR and FRR? Give one example of each type of error in a phone-unlock scenario.

**Answer:** *(write here)*

**Q2.** What happens to FAR and FRR when the threshold increases? Use numbers from Step 8.

**Answer:** *(write here)*

**Q3.** What EER and threshold did you get? What does EER tell you about a system, and what does it *not* tell you?

**Answer:** *(write here)*

**Q4.** Your AUC was close to 1. What does that mean? Would you trust a system with AUC = 0.60?

**Answer:** *(write here)*

**Q5.** Which threshold from Part H (high security or convenience) would you choose for (a) a bank vault and (b) unlocking a personal phone? Justify each choice.

**Answer:** *(write here)*

**Q6.** Based on Part I, how does overlap between genuine and impostor scores affect EER? What could a designer do to reduce the overlap?

**Answer:** *(write here)*

**Q7.** Why is the DET curve usually plotted on log axes instead of linear axes?

**Answer:** *(write here)*

## Deliverables Checklist

- [ ] Public GitHub repository named `biometric-auth-lab1-YOURID`
- [ ] This notebook `Lab1_Biometric_Metrics.ipynb` with **all cells executed** and outputs visible
- [ ] `fig1_distributions.png`, `fig2_far_frr.png`, `fig3_roc_det.png`
- [ ] `genuine_scores.txt` and `impostor_scores.txt`
- [ ] Observations 1–3 and Questions Q1–Q7 answered
- [ ] `README.md` with results and answers
- [ ] At least 3 commits with meaningful messages

## Grading (100 marks)

| Criterion | Marks |
|---|---|
| GitHub repository: correct name, files present, 3+ clear commits | 15 |
| Code runs and produces correct values (FAR/FRR, EER, AUC, thresholds) | 30 |
| Figures: 3 present with labeled axes, titles, legends | 15 |
| Threshold scenarios (Part H) and overlap experiment (Part I) | 15 |
| Observations and analysis questions: correct, clear, own words | 20 |
| README quality | 5 |

## Troubleshooting

| Problem | Fix |
|---|---|
| `NameError: name 'genuine' is not defined` | You skipped or restarted a cell. Use **Runtime → Restart and run all**. |
| `FileNotFoundError` for `genuine_scores.txt` | Run the Step 3 cell first. Colab deletes files when the session ends. |
| Numbers differ from expected output | Check the seed is `42` in Step 3 and you did not change `scale` or `size`. |
| Git rejects your password | Use a Personal Access Token, or use Method A (browser upload). |
| `git push` says branch `main` does not exist | Run `git branch -M main` then `git push -u origin main`. |

*Academic integrity: you may discuss ideas with classmates, but your notebook, results and written answers must be your own work.*